# 🏛️ Legal NLP Pipeline: Indian Judgment Summarization & Question Answering

**BTech Project — Production-Quality NLP Pipeline**

This notebook implements an end-to-end NLP pipeline for processing Indian legal judgments using state-of-the-art transformer models:

| Task | Model | Baseline Metric |
|------|-------|----------------|
| **Summarization** | `facebook/bart-large-cnn` | ROUGE-L: 0.2775 |
| **Question Answering** | `deepset/roberta-base-squad2` | F1: 0.4044 |

**Dataset:** 200 Real Indian Supreme Court & High Court Judgments

---

## 🧩 Cell 1: Environment Setup
Prepare the Colab runtime with all necessary libraries and GPU verification.

In [ ]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP
# ============================================================
# Install required packages
!pip install -q transformers accelerate evaluate peft rouge-score sentencepiece nltk seaborn scikit-learn

import warnings
warnings.filterwarnings('ignore')

# Core imports
import torch
import numpy as np
import pandas as pd
import json, os, re, time
from collections import Counter

# Hugging Face imports
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering, pipeline,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, T5ForConditionalGeneration,
    T5Tokenizer
)
import evaluate

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown, HTML

# NLTK for sentence tokenization
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU verification
print("=" * 60)
print("🖥️  RUNTIME ENVIRONMENT")
print("=" * 60)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device          : {device}")
print(f"  PyTorch Version : {torch.__version__}")
if torch.cuda.is_available():
    print(f"  CUDA Version    : {torch.version.cuda}")
    print(f"  GPU Name        : {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  GPU Memory      : {gpu_mem:.1f} GB")
else:
    print("  ⚠️  No GPU detected. Go to Runtime > Change runtime type > GPU")
print("=" * 60)
print("✅ Environment setup complete!")

## 🧩 Cell 2: Legal Dataset Loader
Load and validate `legal_dataset.json` (200 Indian judgments) with robust error handling.

In [ ]:
# ============================================================
# CELL 2: LEGAL DATASET LOADER
# ============================================================

def load_legal_dataset(filepath):
    """
    Robust loader for the legal dataset JSON file.
    Handles nested schema: documents[].judgment.{judgment_text, summary, ...}
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    metadata = raw_data.get('metadata', {})
    print("📋 Dataset Metadata:")
    for k, v in metadata.items():
        print(f"   {k}: {v}")
    print()

    documents = raw_data.get('documents', [])
    processed = []
    skipped = 0

    for doc in documents:
        try:
            doc_id = doc.get('doc_id', 'UNKNOWN')
            judgment = doc.get('judgment', {})
            judgment_text = judgment.get('judgment_text', '')
            summary = judgment.get('summary', '')
            qa_pairs = doc.get('qa_pairs', [])

            if not judgment_text or not summary:
                skipped += 1
                continue

            processed.append({
                'doc_id': doc_id,
                'case_number': judgment.get('case_number', 'N/A'),
                'court': judgment.get('court', 'N/A'),
                'judge': judgment.get('judge', 'N/A'),
                'date': judgment.get('date', 'N/A'),
                'case_type': judgment.get('case_type', 'N/A'),
                'legal_provisions': judgment.get('legal_provisions', 'N/A'),
                'judgment_text': judgment_text,
                'summary': summary,
                'outcome': judgment.get('outcome', 'N/A'),
                'qa_pairs': [
                    {'question': qa.get('question',''), 'answer': qa.get('answer',''),
                     'question_type': qa.get('question_type','unknown')}
                    for qa in qa_pairs if qa.get('question') and qa.get('answer')
                ]
            })
        except Exception as e:
            print(f"⚠️  Error processing document: {e}")
            skipped += 1
            continue

    print(f"✅ Loaded {len(processed)} documents ({skipped} skipped)")
    return processed

# --- Load the dataset ---
DATASET_PATH = 'legal_dataset.json'
if not os.path.exists(DATASET_PATH):
    from google.colab import files
    print("📁 Please upload legal_dataset.json:")
    uploaded = files.upload()
    DATASET_PATH = list(uploaded.keys())[0]

dataset = load_legal_dataset(DATASET_PATH)

# --- Tokenizer for statistics ---
tokenizer_stats = AutoTokenizer.from_pretrained('facebook/bart-large-cnn')

# --- Text length statistics ---
print("\n" + "=" * 60)
print("📊 TEXT LENGTH STATISTICS (BART Tokens)")
print("=" * 60)

judgment_lengths = [len(tokenizer_stats.encode(d['judgment_text'])) for d in dataset]
summary_lengths = [len(tokenizer_stats.encode(d['summary'])) for d in dataset]

stats_df = pd.DataFrame({
    'Metric': ['Min', 'Max', 'Mean', 'Median', 'Std Dev'],
    'Judgment Tokens': [min(judgment_lengths), max(judgment_lengths),
        f"{np.mean(judgment_lengths):.0f}", f"{np.median(judgment_lengths):.0f}",
        f"{np.std(judgment_lengths):.0f}"],
    'Summary Tokens': [min(summary_lengths), max(summary_lengths),
        f"{np.mean(summary_lengths):.0f}", f"{np.median(summary_lengths):.0f}",
        f"{np.std(summary_lengths):.0f}"]
})
display(stats_df)

# --- QA pair statistics ---
qa_counts = [len(d['qa_pairs']) for d in dataset]
print(f"\n📝 QA Pairs: Total = {sum(qa_counts)}, Per doc avg = {np.mean(qa_counts):.1f}")

# --- Train/Validation Split (80/20) ---
from sklearn.model_selection import train_test_split
try:
    court_labels = [d['court'] for d in dataset]
    train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=SEED, stratify=court_labels)
except ValueError:
    train_data, val_data = train_test_split(dataset, test_size=0.2, random_state=SEED)

print(f"\n📂 Split: Train = {len(train_data)}, Validation = {len(val_data)}")

# --- Sanity check ---
print("\n" + "=" * 60)
print("📄 SAMPLE DOCUMENT")
print("=" * 60)
sample = dataset[0]
print(f'Doc ID: {sample["doc_id"]}')
print(f'Case: {sample["case_number"]}')
print(f'Court: {sample["court"]}')
print(f'Judge: {sample["judge"]}')
print(f'\nJudgment (first 500 chars):\n{sample["judgment_text"][:500]}...')
print(f'\nGround Truth Summary:\n{sample["summary"][:300]}...')
print(f'\nQA Pairs: {len(sample["qa_pairs"])}')
print("✅ Dataset loaded and validated!")

## 🧩 Cell 3: Optimized Legal Summarizer
Improve summarization using advanced decoding strategies with `facebook/bart-large-cnn`.

In [ ]:
# ============================================================
# CELL 3: OPTIMIZED LEGAL SUMMARIZER
# ============================================================

rouge_metric = evaluate.load('rouge')

print("🔄 Loading facebook/bart-large-cnn...")
summ_model_name = "facebook/bart-large-cnn"
summ_tokenizer = AutoTokenizer.from_pretrained(summ_model_name)
summ_model = AutoModelForSeq2SeqLM.from_pretrained(summ_model_name).to(device)
summ_model.eval()
print(f"✅ Model loaded on {device}")

def generate_summary(text, model, tokenizer, strategy='baseline'):
    """Generate summary with baseline or optimized decoding."""
    inputs = tokenizer(text, return_tensors='pt', max_length=1024, truncation=True, padding=True).to(device)
    with torch.no_grad():
        if strategy == 'baseline':
            outputs = model.generate(**inputs, max_length=150, min_length=30)
        else:
            outputs = model.generate(
                **inputs, num_beams=5, length_penalty=1.5,
                min_length=80, max_length=300,
                early_stopping=True, no_repeat_ngram_size=3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "=" * 60)
print("📝 RUNNING SUMMARIZATION (Validation Set)")
print("=" * 60)

baseline_summaries = []
optimized_summaries = []
reference_summaries = []

for i, doc in enumerate(val_data):
    if (i + 1) % 10 == 0:
        print(f"  Processing {i+1}/{len(val_data)}...")
    reference_summaries.append(doc['summary'])
    baseline_summaries.append(generate_summary(doc['judgment_text'], summ_model, summ_tokenizer, 'baseline'))
    optimized_summaries.append(generate_summary(doc['judgment_text'], summ_model, summ_tokenizer, 'optimized'))

baseline_rouge = rouge_metric.compute(predictions=baseline_summaries, references=reference_summaries)
optimized_rouge = rouge_metric.compute(predictions=optimized_summaries, references=reference_summaries)

print("\n📊 ROUGE SCORES")
print("-" * 50)
rouge_results = pd.DataFrame({
    'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Baseline': [f"{baseline_rouge['rouge1']:.4f}", f"{baseline_rouge['rouge2']:.4f}", f"{baseline_rouge['rougeL']:.4f}"],
    'Optimized Decoding': [f"{optimized_rouge['rouge1']:.4f}", f"{optimized_rouge['rouge2']:.4f}", f"{optimized_rouge['rougeL']:.4f}"]
})
display(rouge_results)

summ_scores = {
    'baseline_rougeL': baseline_rouge['rougeL'],
    'optimized_rougeL': optimized_rouge['rougeL'],
}
print("✅ Summarization complete!")

## 🧩 Cell 4: Optimized QA Engine (Long Context Handling)
Enable QA over long Indian judgments (>512 tokens) using sliding window with `deepset/roberta-base-squad2`.

In [ ]:
# ============================================================
# CELL 4: OPTIMIZED QA ENGINE (SLIDING WINDOW)
# ============================================================

print("🔄 Loading deepset/roberta-base-squad2...")
qa_model_name = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name).to(device)
qa_model.eval()
print(f"✅ QA Model loaded on {device}")

def sliding_window_qa(question, context, model, tokenizer, window_size=384, stride=128):
    """Perform extractive QA with sliding window for long contexts."""
    context_tokens = tokenizer.encode(context, add_special_tokens=False)
    question_tokens = tokenizer.encode(question, add_special_tokens=False)
    max_ctx = window_size - len(question_tokens) - 3

    best_answer = ""
    best_confidence = -1.0

    for start in range(0, len(context_tokens), stride):
        chunk = context_tokens[start:start + max_ctx]
        if len(chunk) == 0:
            break

        input_ids = [tokenizer.cls_token_id] + question_tokens + [tokenizer.sep_token_id] + chunk + [tokenizer.sep_token_id]
        attention_mask = [1] * len(input_ids)

        pad_len = window_size - len(input_ids)
        if pad_len > 0:
            input_ids += [tokenizer.pad_token_id] * pad_len
            attention_mask += [0] * pad_len

        inputs = {
            'input_ids': torch.tensor([input_ids]).to(device),
            'attention_mask': torch.tensor([attention_mask]).to(device),
        }

        with torch.no_grad():
            outputs = model(**inputs)

        ctx_start = len(question_tokens) + 2
        ctx_end = ctx_start + len(chunk)

        start_probs = torch.softmax(outputs.start_logits[0][ctx_start:ctx_end], dim=0)
        end_probs = torch.softmax(outputs.end_logits[0][ctx_start:ctx_end], dim=0)

        s_idx = torch.argmax(start_probs).item()
        e_idx = torch.argmax(end_probs).item()

        if e_idx >= s_idx:
            conf = (start_probs[s_idx] * end_probs[e_idx]).item()
            ans = tokenizer.decode(chunk[s_idx:e_idx + 1], skip_special_tokens=True)
            if conf > best_confidence:
                best_confidence = conf
                best_answer = ans

    return best_answer.strip(), best_confidence

def compute_f1(prediction, ground_truth):
    """Compute token-level F1 score."""
    pred_tokens = prediction.lower().split()
    gt_tokens = ground_truth.lower().split()
    if not pred_tokens or not gt_tokens:
        return float(pred_tokens == gt_tokens)
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)

print("\n" + "=" * 60)
print("❓ RUNNING QA ENGINE (Validation Set)")
print("=" * 60)

qa_results_baseline = []
qa_results_sliding = []
qa_pipe = pipeline('question-answering', model=qa_model, tokenizer=qa_tokenizer, device=device)

total_pairs = 0
for i, doc in enumerate(val_data):
    if (i + 1) % 10 == 0:
        print(f"  Processing doc {i+1}/{len(val_data)}...")
    for qa in doc['qa_pairs']:
        total_pairs += 1
        question = qa['question']
        context = doc['judgment_text']
        gt = qa['answer']

        try:
            bl = qa_pipe(question=question, context=context[:2000])
            qa_results_baseline.append(compute_f1(bl['answer'], gt))
        except Exception:
            qa_results_baseline.append(0.0)

        try:
            sw_ans, sw_conf = sliding_window_qa(question, context, qa_model, qa_tokenizer)
            qa_results_sliding.append({'f1': compute_f1(sw_ans, gt), 'confidence': sw_conf, 'answer': sw_ans, 'ground_truth': gt, 'question': question})
        except Exception:
            qa_results_sliding.append({'f1': 0.0, 'confidence': 0.0, 'answer': '', 'ground_truth': gt, 'question': question})

baseline_f1_avg = np.mean(qa_results_baseline)
sliding_f1_avg = np.mean([r['f1'] for r in qa_results_sliding])

print(f"\n📊 QA RESULTS ({total_pairs} QA pairs)")
print("-" * 50)
qa_df = pd.DataFrame({'Method': ['Baseline (Truncated)', 'Sliding Window'], 'Avg F1': [f'{baseline_f1_avg:.4f}', f'{sliding_f1_avg:.4f}']})
display(qa_df)

qa_scores = {'baseline_f1': baseline_f1_avg, 'sliding_f1': sliding_f1_avg}
print("✅ QA Engine evaluation complete!")

## 🧩 Cell 5: Comparison & Visualization View
Human-readable evaluation with side-by-side comparison of summaries.

In [ ]:
# ============================================================
# CELL 5: COMPARISON & VISUALIZATION VIEW
# ============================================================

def display_comparison(doc, bl_summ, opt_summ, idx):
    """Display a clean comparison for a document."""
    border = 'border:2px solid #4a90d9; border-radius:12px; padding:20px; margin:15px 0; background:#f8f9fa;'
    html_parts = []
    html_parts.append(f'<div style="{border}">')
    html_parts.append(f'<h3 style="color:#2c3e50;">📄 Document {idx+1}: {doc["case_number"]}</h3>')
    html_parts.append(f'<p><b>Court:</b> {doc["court"]} | <b>Judge:</b> {doc["judge"]} | <b>Outcome:</b> {doc["outcome"]}</p>')
    jt = doc['judgment_text'][:1500]
    html_parts.append(f'<details><summary style="cursor:pointer; color:#2980b9; font-weight:bold;">📜 Original Judgment (click to expand)</summary><div style="background:#fff; padding:12px; border-radius:8px; max-height:200px; overflow-y:auto; font-size:13px;">{jt}...</div></details>')
    html_parts.append('<div style="display:grid; grid-template-columns:1fr 1fr 1fr; gap:12px; margin-top:12px;">')
    html_parts.append(f'<div style="background:#e8f5e9; padding:12px; border-radius:8px;"><h4 style="color:#2e7d32;">✅ Ground Truth</h4><p style="font-size:13px;">{doc["summary"]}</p></div>')
    html_parts.append(f'<div style="background:#e3f2fd; padding:12px; border-radius:8px;"><h4 style="color:#1565c0;">🔵 Baseline</h4><p style="font-size:13px;">{bl_summ}</p></div>')
    html_parts.append(f'<div style="background:#fff3e0; padding:12px; border-radius:8px;"><h4 style="color:#e65100;">🟠 Optimized</h4><p style="font-size:13px;">{opt_summ}</p></div>')
    html_parts.append('</div></div>')
    display(HTML(''.join(html_parts)))

print("=" * 60)
print("📊 SIDE-BY-SIDE COMPARISON (First 5 Documents)")
print("=" * 60)

num_display = min(5, len(val_data))
for i in range(num_display):
    display_comparison(val_data[i], baseline_summaries[i], optimized_summaries[i], i)

print("✅ Comparison view generated!")

## 🚀 Cell 6: Domain-Specific Fine-Tuning (Summarization)
Adapt BART to Indian legal language with mixed precision training.

**Focus vocabulary:** Petitioner, Respondent, Affidavit, Writ Petition, Final Order

In [ ]:
# ============================================================
# CELL 6: DOMAIN-SPECIFIC FINE-TUNING
# ============================================================
from torch.utils.data import Dataset as TorchDataset

class LegalSummarizationDataset(TorchDataset):
    """Custom dataset for legal summarization fine-tuning."""
    def __init__(self, data, tokenizer, max_input_len=1024, max_target_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        doc = self.data[idx]
        inputs = self.tokenizer(doc['judgment_text'], max_length=self.max_input_len, truncation=True, padding='max_length', return_tensors='pt')
        targets = self.tokenizer(doc['summary'], max_length=self.max_target_len, truncation=True, padding='max_length', return_tensors='pt')
        labels = targets['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'input_ids': inputs['input_ids'].squeeze(), 'attention_mask': inputs['attention_mask'].squeeze(), 'labels': labels}

print("📦 Preparing fine-tuning datasets...")
train_dataset = LegalSummarizationDataset(train_data, summ_tokenizer)
val_dataset = LegalSummarizationDataset(val_data, summ_tokenizer)
print(f"  Train: {len(train_dataset)} | Validation: {len(val_dataset)}")

# Training Arguments (Colab-safe)
training_args = Seq2SeqTrainingArguments(
    output_dir='./legal_bart_finetuned',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    predict_with_generate=True,
    generation_max_length=300,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    report_to='none',
    seed=SEED,
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    labels = np.where(labels != -100, labels, summ_tokenizer.pad_token_id)
    decoded_preds = summ_tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = summ_tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}

data_collator = DataCollatorForSeq2Seq(summ_tokenizer, model=summ_model, padding=True)

print("\n🚀 Starting fine-tuning...")
print("   Epochs: 3 | Batch: 2 | Grad Accum: 8 | LR: 2e-5 | FP16:", torch.cuda.is_available())

trainer = Seq2SeqTrainer(
    model=summ_model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    tokenizer=summ_tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print(f"\n✅ Fine-tuning complete! Training Loss: {train_result.training_loss:.4f}")

# Evaluate fine-tuned model
print("\n📊 Evaluating fine-tuned model on validation set...")
finetuned_summaries = [generate_summary(doc['judgment_text'], summ_model, summ_tokenizer, 'optimized') for doc in val_data]

finetuned_rouge = rouge_metric.compute(predictions=finetuned_summaries, references=reference_summaries)

print("\n📊 FINE-TUNED ROUGE SCORES")
print("-" * 50)
ft_df = pd.DataFrame({
    'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Before Fine-Tuning': [f"{optimized_rouge['rouge1']:.4f}", f"{optimized_rouge['rouge2']:.4f}", f"{optimized_rouge['rougeL']:.4f}"],
    'After Fine-Tuning': [f"{finetuned_rouge['rouge1']:.4f}", f"{finetuned_rouge['rouge2']:.4f}", f"{finetuned_rouge['rougeL']:.4f}"]
})
display(ft_df)
summ_scores['finetuned_rougeL'] = finetuned_rouge['rougeL']

## 🧠 Cell 7: Post-Processing with Sentence Scoring
Ensure summaries capture the **final verdict/outcome** using heuristic sentence scoring.

In [ ]:
# ============================================================
# CELL 7: POST-PROCESSING WITH SENTENCE SCORING
# ============================================================

BOOST_PATTERNS = {
    r'\bhence\b': 3.0, r'\btherefore\b': 3.0, r'\bthe court held\b': 4.0,
    r'\bpetition is allowed\b': 5.0, r'\bpetition is dismissed\b': 5.0,
    r'\bappeal is allowed\b': 5.0, r'\bappeal is dismissed\b': 5.0,
    r'\bfinal order\b': 4.0, r'\bdisposed of\b': 3.0,
    r'\baccordingly\b': 2.5, r'\bconsequently\b': 2.5,
    r'\bset aside\b': 3.5, r'\bupheld\b': 3.0,
    r'\bstruck down\b': 4.0, r'\bunconstitutional\b': 3.5,
}

PENALIZE_PATTERNS = {
    r'^the facts of the case': -3.0,
    r'^this (writ|criminal|civil)': -1.5,
    r'\bthe primary issue\b': -1.0,
    r'\bchallenges the judgment\b': -2.0,
}

def score_sentence(sentence):
    """Score a sentence based on legal relevance heuristics."""
    score = 1.0
    s = sentence.lower().strip()
    for pat, w in BOOST_PATTERNS.items():
        if re.search(pat, s): score += w
    for pat, w in PENALIZE_PATTERNS.items():
        if re.search(pat, s): score += w
    return max(score, 0.1)

def post_process_summary(model_summary, judgment_text):
    """Enhance model summary by ensuring verdict sentences are included."""
    j_sents = sent_tokenize(judgment_text)
    m_sents = sent_tokenize(model_summary)

    scored = []
    for i, sent in enumerate(j_sents):
        base = score_sentence(sent)
        pos_bonus = (i / max(len(j_sents), 1)) * 2.0
        scored.append((sent, base + pos_bonus))

    scored.sort(key=lambda x: x[1], reverse=True)

    verdict_sents = []
    for sent, sc in scored[:3]:
        is_new = all(compute_f1(sent, ms) < 0.5 for ms in m_sents)
        if is_new and sc > 3.0:
            verdict_sents.append(sent)

    combined = model_summary
    if verdict_sents:
        combined += ' ' + ' '.join(verdict_sents[:2])
    return combined

print("🔧 Applying post-processing to fine-tuned summaries...")
postprocessed_summaries = [post_process_summary(finetuned_summaries[i], doc['judgment_text']) for i, doc in enumerate(val_data)]

postprocessed_rouge = rouge_metric.compute(predictions=postprocessed_summaries, references=reference_summaries)

print("\n📊 POST-PROCESSED ROUGE SCORES")
print("-" * 50)
pp_df = pd.DataFrame({
    'Metric': ['ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Fine-Tuned': [f"{finetuned_rouge['rouge1']:.4f}", f"{finetuned_rouge['rouge2']:.4f}", f"{finetuned_rouge['rougeL']:.4f}"],
    'Post-Processed': [f"{postprocessed_rouge['rouge1']:.4f}", f"{postprocessed_rouge['rouge2']:.4f}", f"{postprocessed_rouge['rougeL']:.4f}"]
})
display(pp_df)
summ_scores['postprocessed_rougeL'] = postprocessed_rouge['rougeL']
print("✅ Post-processing complete!")

## 🔄 Cell 8: Hybrid QA Strategy (Fallback Mechanism)
Improve answer reliability by falling back to generative QA (`google/flan-t5-base`) when extractive confidence is low.

In [ ]:
# ============================================================
# CELL 8: HYBRID QA STRATEGY (FALLBACK)
# ============================================================

CONFIDENCE_THRESHOLD = 0.3

print("🔄 Loading google/flan-t5-base (generative fallback)...")
gen_model_name = "google/flan-t5-base"
gen_tokenizer = T5Tokenizer.from_pretrained(gen_model_name)
gen_model = T5ForConditionalGeneration.from_pretrained(gen_model_name).to(device)
gen_model.eval()
print(f"✅ Fallback model loaded on {device}")

def generative_qa(question, context, model, tokenizer, max_context=1500):
    """Generate an answer using flan-t5-base for complex questions."""
    prompt = 'Based on the following legal judgment, answer the question.\n\n'
    prompt += 'Context: ' + context[:max_context] + '\n\n'
    prompt += 'Question: ' + question + '\n\nAnswer:'
    inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=150, num_beams=4, early_stopping=True, no_repeat_ngram_size=3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def hybrid_qa(question, context, qa_m, qa_t, gen_m, gen_t, threshold=CONFIDENCE_THRESHOLD):
    """Hybrid QA: extractive first, fallback to generative if low confidence."""
    ext_answer, confidence = sliding_window_qa(question, context, qa_m, qa_t)
    if confidence >= threshold and len(ext_answer.strip()) > 0:
        return ext_answer, confidence, 'extractive'
    gen_answer = generative_qa(question, context, gen_m, gen_t)
    return gen_answer, confidence, 'generative'

print("\n" + "=" * 60)
print("🔄 RUNNING HYBRID QA (Validation Set)")
print("=" * 60)

hybrid_results = []
fallback_examples = []
fallback_count = 0

for i, doc in enumerate(val_data):
    if (i + 1) % 10 == 0:
        print(f"  Processing doc {i+1}/{len(val_data)}...")
    for qa in doc['qa_pairs']:
        ans, conf, method = hybrid_qa(qa['question'], doc['judgment_text'], qa_model, qa_tokenizer, gen_model, gen_tokenizer)
        f1 = compute_f1(ans, qa['answer'])
        hybrid_results.append({'f1': f1, 'method': method, 'confidence': conf})
        if method == 'generative':
            fallback_count += 1
            if len(fallback_examples) < 5:
                fallback_examples.append({'question': qa['question'], 'conf': conf, 'gen_answer': ans, 'gt': qa['answer']})

hybrid_f1_avg = np.mean([r['f1'] for r in hybrid_results])

print(f"\n📊 HYBRID QA RESULTS")
print("-" * 50)
print(f"  Total QA Pairs     : {len(hybrid_results)}")
print(f"  Fallback Activated : {fallback_count} ({100*fallback_count/max(len(hybrid_results),1):.1f}%)")
print(f"  Hybrid F1          : {hybrid_f1_avg:.4f}")

hdf = pd.DataFrame({
    'Method': ['Baseline (Truncated)', 'Sliding Window', 'Hybrid (SW + Generative)'],
    'Avg F1': [f'{qa_scores["baseline_f1"]:.4f}', f'{qa_scores["sliding_f1"]:.4f}', f'{hybrid_f1_avg:.4f}']
})
display(hdf)

if fallback_examples:
    print("\n📝 FALLBACK EXAMPLES:")
    print("-" * 50)
    for i, ex in enumerate(fallback_examples[:3]):
        print(f'\nExample {i+1}:')
        print(f'  Question: {ex["question"]}')
        print(f'  Extractive Confidence: {ex["conf"]:.4f} (below {CONFIDENCE_THRESHOLD})')
        print(f'  Generative Answer: {ex["gen_answer"]}')
        print(f'  Ground Truth: {ex["gt"]}')

qa_scores['hybrid_f1'] = hybrid_f1_avg
print("\n✅ Hybrid QA evaluation complete!")

## 📊 Cell 9: Performance Dashboard
Visual comparison of all improvements across summarization and QA tasks.

In [ ]:
# ============================================================
# CELL 9: PERFORMANCE DASHBOARD
# ============================================================

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('🏛️ Legal NLP Pipeline — Performance Dashboard', fontsize=18, fontweight='bold', y=1.02)

# --- Summarization ROUGE-L ---
ax1 = axes[0]
s_methods = ['Baseline\n(Default)', 'Optimized\nDecoding', 'Fine-Tuned', 'Post-\nProcessed']
s_vals = [summ_scores['baseline_rougeL'], summ_scores['optimized_rougeL'], summ_scores['finetuned_rougeL'], summ_scores['postprocessed_rougeL']]
s_colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']
bars1 = ax1.bar(s_methods, s_vals, color=s_colors, edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars1, s_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax1.set_ylabel('ROUGE-L Score', fontsize=13)
ax1.set_title('📝 Summarization Performance', fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(s_vals) * 1.2)
ax1.axhline(y=0.2775, color='red', linestyle='--', alpha=0.7, label='Original Baseline (0.2775)')
ax1.legend(fontsize=10)

# --- QA F1 ---
ax2 = axes[1]
q_methods = ['Baseline\n(Truncated)', 'Sliding\nWindow', 'Hybrid\n(+ Generative)']
q_vals = [qa_scores['baseline_f1'], qa_scores['sliding_f1'], qa_scores['hybrid_f1']]
q_colors = ['#95a5a6', '#3498db', '#e74c3c']
bars2 = ax2.bar(q_methods, q_vals, color=q_colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars2, q_vals):
    ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005, f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax2.set_ylabel('F1 Score', fontsize=13)
ax2.set_title('❓ Question Answering Performance', fontsize=14, fontweight='bold')
ax2.set_ylim(0, max(q_vals) * 1.2)
ax2.axhline(y=0.4044, color='red', linestyle='--', alpha=0.7, label='Original Baseline (0.4044)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('performance_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Summary Table ---
print("\n" + "=" * 60)
print("📋 FINAL PERFORMANCE SUMMARY")
print("=" * 60)

final_df = pd.DataFrame({
    'Task': ['Summarization']*4 + ['Question Answering']*3,
    'Method': ['Baseline (Default Decoding)', 'Optimized Decoding', 'Fine-Tuned (3 epochs)', 'Post-Processed',
              'Baseline (Truncated)', 'Sliding Window', 'Hybrid (+ Generative)'],
    'Score': [
        f"ROUGE-L: {summ_scores['baseline_rougeL']:.4f}",
        f"ROUGE-L: {summ_scores['optimized_rougeL']:.4f}",
        f"ROUGE-L: {summ_scores['finetuned_rougeL']:.4f}",
        f"ROUGE-L: {summ_scores['postprocessed_rougeL']:.4f}",
        f"F1: {qa_scores['baseline_f1']:.4f}",
        f"F1: {qa_scores['sliding_f1']:.4f}",
        f"F1: {qa_scores['hybrid_f1']:.4f}",
    ]
})
display(final_df)

print("\n✅ Pipeline complete! All results saved.")
print("📊 Dashboard saved to: performance_dashboard.png")